# Задача 4. Пайплайн прогнозирования

Offline-пайплайн `SolarForecastPipeline` (`src/pipeline.py`).

**Выбор модели:** `AutoTheta` — лучший средний RMSE в rolling backtest (stats, h=24, 5 окон). На hold-out 48 ч лидирует `AutoETS` (RMSE ≈ 8704); обе модели поддерживаются пайплайном.

**Этапы:**
1. Подготовка данных (`build_datasets`) → `__output__/train.parquet`, `test.parquet`
2. Обучение statsforecast-модели на train
3. Прогноз на 48 ч для серий Plant 1 и Plant 2
4. Метрики на hold-out, сохранение артефактов

In [ ]:
import json
from pathlib import Path

import pandas as pd

from src.config import DEFAULT_MODEL, OUTPUT_DIR
from src.data import build_datasets, load_train_test
from src.pipeline import SolarForecastPipeline

In [ ]:
train_df, test_df = build_datasets(save=True)
print(f'Train: {len(train_df)} rows, Test: {len(test_df)} rows')
print(f'Series: {train_df["unique_id"].unique().tolist()}')

pipe = SolarForecastPipeline(model_name=DEFAULT_MODEL)
result = pipe.run(train_df=train_df, test_df=test_df)

print('Model:', result.model_name)
print('Metrics:', result.metrics)
print(f'Fit: {result.fit_seconds:.2f}s, Predict: {result.predict_seconds:.3f}s')
result.metrics

In [ ]:
configs = ['Naive', 'SeasonalNaive', 'AutoTheta', 'AutoETS', 'AutoARIMA']
bench = []
for name in configs:
    r = SolarForecastPipeline(model_name=name).run(train_df=train_df, test_df=test_df)
    bench.append({'model': name, **r.metrics, 'fit_s': r.fit_seconds, 'predict_s': r.predict_seconds})

bench_df = pd.DataFrame(bench).set_index('model').sort_values('RMSE')
bench_df

In [ ]:
artifact_dir = OUTPUT_DIR / 'pipeline'
print('Artifacts:')
for p in sorted(artifact_dir.glob('*')):
    print('-', p.name)
    if p.suffix == '.json':
        print(json.loads(p.read_text(encoding='utf-8')))

### Обоснование настройки пайплайна

| Компонент | Выбор | Причина |
|-----------|-------|---------|
| Модель по умолчанию | `AutoTheta` | Минимальный средний RMSE в rolling CV (stats) |
| Альтернатива | `AutoETS` | Лучший RMSE на финальном hold-out 48 ч |
| Горизонт | 48 ч | Постановка задачи (2 суток) |
| Частота | 1 ч | Баланс детализации и сезонности |
| Train/test | Последние 48 ч hold-out | Offline batch, без утечки |

ML/DL исследованы в `2_dl_models.ipynb`, но не включены в production-пайплайн из-за большего времени обучения при сопоставимом качестве на коротком ряде.